# Canopy introduction

## Overview

This tutorial provides an overview of the different ways of specifying canopies in Eradiate.

## Prerequisites

This tutorial assumes basic familiarity with Eradiate workflows, as well as basics of plotting results with matplotlib.

## What you will learn

In this tutorial you will see the different ways of specifying canopies and their constituents. This includes abstract canopies based on floating disks, their arrangement into different geometric shapes, such as cube, cone and ellipsoid (sphere), and defining those same abstract canopies with ASCII files. Canopies based on triangulated meshes are not covered in this tutorial as the distribution of mesh files is impractical for the purposes of this tutorial.

--- 

## Imports
As for all tutorials the first step is to import all necessary classes and create aliases for convenience

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import eradiate
eradiate.set_mode("mono")

import eradiate.scenes as ertsc
from eradiate import unit_registry as ureg

# We activate a few convenient presets
eradiate.plot.set_style(rc={"figure.dpi": 96})

We set the computational mode to monochromatic, since we will be simulating scenes without atmospheres. Hence the correlated k-distribution method will not be useful here.

Next we set up convenience functions for plotting BRF results and visualizing camera renders.

In [ ]:
def show_camera(
    exp,
    measure_id,
    robust=True,
    add_colorbar=False,
    vmin=None,
    vmax=None,
):
    """
    Display the output of a monochromatic camera measure.
    """
    _, ax = plt.subplots(1,1)
    exp.results[measure_id]["radiance"].squeeze(drop=True).plot.imshow(
        ax=ax,
        origin="upper",
        cmap="Greys_r",
        vmin=vmin,
        vmax=vmax,
        robust=robust,
        add_colorbar=add_colorbar,
    )
    ax.set_aspect(1)
    plt.show()
    plt.close()

def show_brf(exp, measure_id):
    """
    Display the BRF output of a distant radiance measure.
    """
    _, ax = plt.subplots(1,1)
    with plt.style.context({"lines.linestyle": ":", "lines.marker": "."}):
        exp.results[measure_id]["brf"].squeeze(drop=True).plot(ax=ax, x="vza")
    plt.show()
    plt.close()

## Creating the scene step by step

We start by creating a simple scene consisting only in an illuminated lambertian surface. As opposed to other tutorials, here we have to specify the horizontal extent of the surface, because no other scene elements which can act as guidance exist yet.
Note the use of the Eradiate unit registry `ureg` to attach the unit *meter* to the surface's edge length.

Next we create a perspective camera to visualize the surface.

In [ ]:
lambertian_surface=ertsc.surface.BasicSurface(
    shape=ertsc.shapes.RectangleShape(edges=10.0 * ureg.m),
    bsdf=ertsc.bsdfs.LambertianBSDF(reflectance=0.5),
)

camera_oblique = ertsc.measure.PerspectiveCameraMeasure(
    id="camera_oblique",
    origin=[15, 15, 15] * ureg.m,
    target=[0, 0, 0] * ureg.m,
    up=[0, 0, 1],
    film_resolution=(320, 240),
    spp=512,
)

Now we create the experiment object. We are using the [RamiExperiment](../rst/reference_api/generated/autosummary/eradiate.experiments.RamiExperiment.rst) class, which is taylored to scenes with 3D canopies without atmospheres. We specify neither a canopy, nor an illumination. The canopy's default is simply its absence, while the default illumination is a directional emitter with both zenith and azimuth angle equal to 0.

We run the simulation and use the convenience function defined above to visualize the result.

In [ ]:
exp = eradiate.experiments.CanopyExperiment(
    surface=lambertian_surface,
    measures=camera_oblique,
)

In [ ]:
eradiate.run(exp)
show_camera(exp, "camera_oblique")

## Adding a canopy

Now it is time to add a canopy. We start with the simplest canopy possible in Eradiate. A homogeneous cloud of floating disks. The [DiscreteCanopy](../rst/reference_api/generated/autosummary/eradiate.scenes.biosphere.DiscreteCanopy.rst) class has a convenience constructor method [homogeneous()](../rst/reference_api/generated/autosummary/eradiate.scenes.biosphere.DiscreteCanopy.rst#eradiate.scenes.biosphere.DiscreteCanopy.homogeneous), which is ideal for this case.
Note how the unit registry allows for intuitive numerical values for both the horizontal extent of the canopy as well as the radius of the floating leaves.

In [ ]:
homogeneous_canopy = ertsc.biosphere.DiscreteCanopy.homogeneous(
    l_vertical=1.0 * ureg.m,
    l_horizontal=10.0 * ureg.m,
    lai=2.0,
    leaf_radius=10 * ureg.cm,
)

We create a new experiment object, which contains the canopy we just defined. Note that we can now define the surface through its BSDF only, because the size of the rectangular shape is defined by the width of the canopy.

We run the experiment and print the result.

In [ ]:
exp = eradiate.experiments.CanopyExperiment(
    surface=ertsc.bsdfs.LambertianBSDF(reflectance=0.5),
    canopy=homogeneous_canopy,
    measures=camera_oblique,
)
eradiate.run(exp)
show_camera(exp, "camera_oblique")

### More options
Of course Eradiate supports more complex geometries for canopies. By manually defining the constituents of the `DiscreteCanopy`, we can exert more detailed control over them.

This reveals some of the inner workings of the `DiscreteCanopy`. The canopy holds a list of [InstancedCanopyElement](../rst/reference_api/generated/autosummary/eradiate.scenes.biosphere.DiscreteCanopy.rst#eradiate.scenes.biosphere.InstancedCanopyElement) objects, which act as a container for explicit canopy elements such as the [LeafCloud](../rst/reference_api/generated/autosummary/eradiate.scenes.biosphere.DiscreteCanopy.rst#eradiate.scenes.biosphere.LeafCloud) object we created and a list of positions, at which instances or clones of the canopy element will be placed.

We define only one instance position that places the spherical canopy on top of the surface in the experiment.

In [ ]:
spherical_leaf_cloud = ertsc.biosphere.LeafCloud.sphere(
    radius = 5 * ureg.m,
    n_leaves = 10000,
    leaf_radius = 10 * ureg.cm,
    leaf_reflectance=0.5,
    leaf_transmittance=0.5
)
canopy_sphere = ertsc.biosphere.DiscreteCanopy(
    instanced_canopy_elements=ertsc.biosphere.InstancedCanopyElement(
        id="sphere",
        canopy_element=spherical_leaf_cloud, 
        instance_positions=[(0, 0, 5.0)]*ureg.m
    ),
    size=[10.0, 10.0, 10.0]*ureg.m
)

Since this canopy has considerable extent in the vertical direction, we will update the camera plugin, such that the canopy is well within the frame.

In [ ]:
camera_oblique = ertsc.measure.PerspectiveCameraMeasure(
    id="camera_oblique",
    origin=[15, 15, 15] * ureg.m,
    target=[0, 0, 3.0] * ureg.m,
    up=[0, 0, 1],
    film_resolution=(320, 240),
    spp=512,
)

Again, we create the experiment,run it and visualize the results.

In [ ]:
exp = eradiate.experiments.RamiExperiment(
    surface=ertsc.bsdfs.LambertianBSDF(reflectance=0.8),
    canopy=canopy_sphere,
    measures=camera_oblique,
)
eradiate.run(exp)
show_camera(exp, "camera_oblique")

For this kind of parametrization, constructor methods are available for ellipsoid and conical shapes as well.

## Creating a canopy from definition files

### **WARNING:** This section will create files on your machine, be sure to change file paths according to your local data structure or move files that might be overwritten!

In order for a simulation to be reproducible users may not want to rely on the stochastic placement of leaves as it was used above. For this reason Eradiate canopies can be specified using ASCII files.

To demonstrate this capability we first have to create two files. One file will specify the placement of the leaves inside the leaf cloud, the other will specify the positions at which the leaf cloud will be positioned (This is the `instance_positions` parameter from before). 

The ASCII definition file for the leaf cloud has to be formatted as follows:

- One leaf per line
- Each line holds seven floating point values
    - leaf radius in centimeters
    - x, y and z coordinate of the leaf center position
    - x, y and z coordinate of the leaf normal vector
    
We will create a file with 5000 leaves at the root of the current working directory.

The ASCII file for the instance positions simply holds three floating point values per line, which specify the x, y and z coordinates for the instances.

Note that all numbers in these files will be interpreted to be in meters!

In [ ]:
fname_cloud = "./tutorial_canopy_definition.txt"
fname_instances = "./tutorial_canopy_instances.txt"

with open(fname_cloud, "w") as outfile:
    for i in range(5000):
        rands = np.random.random(6)
        px = (rands[0] - 0.5) * 2 * 2
        py = (rands[1] - 0.5) * 2 * 2
        pz = (rands[2] - 0.5) * 2 * 2
        
        nx = (rands[3] - 0.5) * 2
        ny = (rands[4] - 0.5) * 2
        nz = (rands[5] - 0.5) * 2
        
        outfile.write(f"0.05 {px} {py} {pz} {nx} {ny} {nz}\n")
        
with open(fname_instances, "w") as outfile:
    outfile.write("-2.0 2.0 2.0\n")
    outfile.write("2.0 -2.0 2.0\n")

The instance positions we defined in the last file should be positioning two copies of the cuboid leaf cloud positioned diagonally with respect to each other.

Now we create a canopy that uses these definition files. Here we can use another convenience constructor of the `DiscreteCanopy` class.

In [ ]:
canopy_files = ertsc.biosphere.DiscreteCanopy.leaf_cloud_from_files(
    id="leaf_cloud",
    leaf_cloud_dicts = [
        {
            "instance_filename": fname_instances,
            "leaf_cloud_filename": fname_cloud,
            "leaf_reflectance": 0.5,
            "leaf_transmittance": 0.5,
            "sub_id": "cube"
        }
    ],
    size=(8, 8, 4)*ureg.m
)

Again we update our camera to properly show our canopy.

In [ ]:
camera_oblique = ertsc.measure.PerspectiveCameraMeasure(
    id="camera_oblique",
    origin=[10, 10, 10] * ureg.m,
    target=[0, 0, 0.0] * ureg.m,
    up=[0, 0, 1],
    film_resolution=(640, 480),
    spp=128,
)

In [ ]:
exp = eradiate.experiments.RamiExperiment(
    surface=ertsc.bsdfs.LambertianBSDF(reflectance=0.8),
    canopy=canopy_files,
    measures=camera_oblique,
)
eradiate.run(exp)
show_camera(exp, "camera_oblique")

## Final words
Eradiate supports the specification of canopies through obj based mesh files, which allows for a more realistic approximation of real world canopies. However, distributing mesh files with these tutorials is impractical and therefore this method will not be demonstrated in detail. Instead interested users are referred to the documentation of the [MeshTree](../rst/reference_api/generated/autosummary/eradiate.scenes.biosphere.DiscreteCanopy.rst#eradiate.scenes.biosphere.MeshTree) class.